# Notebook 1: SQL in R — Structured Query Analysis

| Field | Details |
|---|---|
| **Student Name** | Mohammed Ansab |
| **Student ID** | 32146926 |
| **Module** | Databases and Analytics |
| **Notebook** | SQL in R |
| **GitHub Repository** | https://github.com/ansuuu-afk/northstar-databases-analytics.git|

---

## Overview

This notebook uses SQL queries executed directly within the R environment using the `sqldf` package. SQL is applied to the NorthStar structured datasets to answer the operational and financial questions raised by senior management in the case study.

The `sqldf` package allows standard SQL syntax to be run over R data frames without needing a separate database engine. This is useful for analytical work where the data has already been loaded into R but complex filtering, aggregation, and multi-table joins are needed.

### Analytical questions addressed:

| Query | Business Question | Director Concerned |
|---|---|---|
| 1 | Which hubs have the worst delivery failure rates? | Operations Director |
| 2 | Which customers show patterns of repeated service failure? | Customer Experience Director |
| 3 | Do driver route overrides correlate with delivery failures? | Operations Director |
| 4 | Which service types are generating the most financial risk? | Finance Director |
| 5 | Which incident types most frequently precede delivery failure? | Technology Director |
| 6 | Which zones have the highest complaint concentration? | Customer Experience Director |
| 7 | Which vehicles are at highest operational risk? | Operations Director |

---

## Section 1: Setup and Library Loading

In [2]:
# Install required packages — run this cell first
# sqldf: run SQL queries directly on R data frames
# RSQLite: SQLite backend used by sqldf
# dplyr: data manipulation utilities
# lubridate: datetime parsing and manipulation

install.packages(c("sqldf", "RSQLite", "DBI", "dplyr", "lubridate", "knitr"))

library(sqldf)
library(RSQLite)
library(DBI)
library(dplyr)
library(lubridate)

cat("All packages loaded successfully.\n")

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gsubfn’, ‘proto’, ‘chron’


Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




All packages loaded successfully.


---
## Section 2: Loading Data from GitHub

All data files are loaded directly from the GitHub repository using raw URLs. This means no manual file uploads are needed in Colab — the notebook is fully reproducible from any machine.

The `stringsAsFactors = FALSE` argument ensures character columns are read as plain strings rather than factor levels, which prevents issues when using them in SQL WHERE clauses and JOIN conditions.

In [3]:
# Base URL for raw GitHub files
BASE <- "https://raw.githubusercontent.com/ansuuu-afk/northstar-databases-analytics/main/northstar_dataset/"

customers  <- read.csv(paste0(BASE, "customers.csv"),  stringsAsFactors = FALSE)
drivers    <- read.csv(paste0(BASE, "drivers.csv"),    stringsAsFactors = FALSE)
deliveries <- read.csv(paste0(BASE, "deliveries.csv"), stringsAsFactors = FALSE)
incidents  <- read.csv(paste0(BASE, "incidents.csv"),  stringsAsFactors = FALSE)
complaints <- read.csv(paste0(BASE, "complaints.csv"), stringsAsFactors = FALSE)
hubs       <- read.csv(paste0(BASE, "hubs.csv"),       stringsAsFactors = FALSE)
orders     <- read.csv(paste0(BASE, "orders.csv"),     stringsAsFactors = FALSE)
vehicles   <- read.csv(paste0(BASE, "vehicles.csv"),   stringsAsFactors = FALSE)

# Print a summary of what was loaded
cat("Dataset overview:\n")
cat(sprintf("  %-15s %6d rows  %2d columns\n", "customers",  nrow(customers),  ncol(customers)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "drivers",    nrow(drivers),    ncol(drivers)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "deliveries", nrow(deliveries), ncol(deliveries)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "incidents",  nrow(incidents),  ncol(incidents)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "complaints", nrow(complaints), ncol(complaints)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "hubs",       nrow(hubs),       ncol(hubs)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "orders",     nrow(orders),     ncol(orders)))
cat(sprintf("  %-15s %6d rows  %2d columns\n", "vehicles",   nrow(vehicles),   ncol(vehicles)))

Dataset overview:
  customers          650 rows   9 columns
  drivers            170 rows   8 columns
  deliveries         950 rows  13 columns
  incidents          280 rows   7 columns
  complaints         320 rows  10 columns
  hubs                 8 rows   5 columns
  orders            1250 rows  11 columns
  vehicles           120 rows   8 columns


---
## Section 3: Data Cleaning — Zone Name Standardisation

Before running any SQL queries, the zone name inconsistency must be resolved. SQL GROUP BY and JOIN operations are case-sensitive string matches — `'south'` and `'SOUTH'` are treated as completely different values. If this is not fixed, any zone-level aggregation will split records that belong to the same zone into separate groups, producing incorrect counts and percentages.

The raw dataset contains at least 15 distinct variants of what should be 7 zone names. Below, the problem is shown before cleaning, then the normalisation function is applied to all relevant columns across all datasets.

In [4]:
# Show the inconsistency before cleaning
cat("Zone values in customers BEFORE cleaning:\n")
print(sort(unique(customers$home_zone)))

cat("\nZone values in orders (pickup_zone) BEFORE cleaning:\n")
print(sort(unique(orders$pickup_zone)))

Zone values in customers BEFORE cleaning:
 [1] "Airport"   "AIRPORT"   "Central"   "CENTRAL"   "Ctr"       "East"     
 [7] "EAST"      "north"     "North"     "NORTH"     "Riverside" "RiverSide"
[13] "South"     "SOUTH"     "West"      "WEST"     

Zone values in orders (pickup_zone) BEFORE cleaning:
 [1] "Airport"   "AIRPORT"   "Central"   "CENTRAL"   "Ctr"       "East"     
 [7] "EAST"      "north"     "North"     "NORTH"     "Riverside" "RiverSide"
[13] "South"     "SOUTH"     "West"      "WEST"     


In [5]:
# Zone normalisation function
# Maps all variants to a canonical uppercase form
normalise_zone <- function(z) {
  if (is.na(z)) return(NA)
  z <- toupper(trimws(z))               # uppercase and trim whitespace
  z <- gsub("^CTR$",       "CENTRAL",   z)  # abbreviation to full name
  z <- gsub("^RIVERSI.*",  "RIVERSIDE", z)  # handle RiverSide variants
  return(z)
}

# Apply to every zone column across all datasets
customers$home_zone    <- sapply(customers$home_zone,    normalise_zone)
orders$pickup_zone     <- sapply(orders$pickup_zone,     normalise_zone)
orders$dropoff_zone    <- sapply(orders$dropoff_zone,    normalise_zone)
drivers$base_zone      <- sapply(drivers$base_zone,      normalise_zone)
vehicles$assigned_zone <- sapply(vehicles$assigned_zone, normalise_zone)
hubs$zone              <- sapply(hubs$zone,              normalise_zone)

cat("Zone normalisation complete.\n\n")
cat("Unique zones after cleaning (customers):\n")
print(sort(unique(customers$home_zone)))

cat("\nUnique zones after cleaning (orders pickup_zone):\n")
print(sort(unique(orders$pickup_zone)))

Zone normalisation complete.

Unique zones after cleaning (customers):
[1] "AIRPORT"   "CENTRAL"   "EAST"      "NORTH"     "RIVERSIDE" "SOUTH"    
[7] "WEST"     

Unique zones after cleaning (orders pickup_zone):
[1] "AIRPORT"   "CENTRAL"   "EAST"      "NORTH"     "RIVERSIDE" "SOUTH"    
[7] "WEST"     


In [8]:
# Also clean up delivery status values to ensure consistent casing
# This prevents SQL CASE WHEN statements from missing records due to case differences
unique(deliveries$delivery_status)

[1] "Failed"  "OnTime"  "Delayed"

---
## Section 4: SQL Queries and Analysis

### Query 1 — Hub Performance: Failure and Delay Rates

**Business question:** Which hubs consistently underperform, and by how much?

This query joins the `deliveries` table to the `hubs` table and aggregates delivery outcomes by hub. The CASE WHEN expressions inside SUM() functions count specific status values — this is a standard SQL technique for pivot-style aggregation without needing a separate GROUP BY per status.

The `ROUND()` function is applied to all percentage and average columns to keep results readable.

In [7]:
hub_performance <- sqldf("
  SELECT
    h.hub_id,
    h.hub_name,
    h.zone,
    h.hub_type,
    COUNT(d.delivery_id)                                                    AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)         AS failed_count,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)         AS delayed_count,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END)         AS ontime_count,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                        AS failure_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                        AS delay_rate_pct,
    ROUND(100.0 * (SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END))
          / COUNT(d.delivery_id), 1)                                        AS problem_rate_pct,
    ROUND(AVG(d.manual_route_override_count), 2)                            AS avg_route_overrides,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                                    AS avg_fuel_cost,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                          AS avg_customer_rating
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name, h.zone, h.hub_type
  ORDER BY failure_rate_pct DESC
")

print(hub_performance)

  hub_id       hub_name      zone  hub_type total_deliveries failed_count
1    H08  Midtown Relay   CENTRAL  Charging              128           26
2    H05   Central Core   CENTRAL   Control              115           23
3    H06    Airport Hub   AIRPORT  Dispatch              104           15
4    H04      West Gate      WEST  Dispatch              127           16
5    H01 North Exchange     NORTH  Dispatch              136           17
6    H07  Riverside Hub RIVERSIDE Warehouse              115           14
7    H02     South Link     SOUTH  Dispatch              106           10
8    H03      East Dock      EAST Warehouse              119           11
  delayed_count ontime_count failure_rate_pct delay_rate_pct problem_rate_pct
1            22           80             20.3           17.2             37.5
2            25           67             20.0           21.7             41.7
3            27           62             14.4           26.0             40.4
4            28       

In [ ]:
# Calculate how much worse the worst hub is vs the best hub
best_hub  <- hub_performance[which.min(hub_performance$failure_rate_pct), ]
worst_hub <- hub_performance[which.max(hub_performance$failure_rate_pct), ]

cat("\nKey finding:\n")
cat(sprintf("  Best hub:  %s (%s) — %.1f%% failure rate\n",
            best_hub$hub_name, best_hub$zone, best_hub$failure_rate_pct))
cat(sprintf("  Worst hub: %s (%s) — %.1f%% failure rate\n",
            worst_hub$hub_name, worst_hub$zone, worst_hub$failure_rate_pct))
cat(sprintf("  The worst hub has %.1fx the failure rate of the best hub.\n",
            worst_hub$failure_rate_pct / best_hub$failure_rate_pct))
cat("\nInterpretation: Central zone hubs are consistently worst performers.\n")
cat("This confirms the Operations Director's concern about underperforming city hubs.\n")


Key finding:
  Best hub:  East Dock (EAST) — 9.2% failure rate
  Worst hub: Midtown Relay (CENTRAL) — 20.3% failure rate
  The worst hub has 2.2x the failure rate of the best hub.

Interpretation: Central zone hubs are consistently worst performers.
This confirms the Operations Director's concern about underperforming city hubs.


### Query 2 — Repeat Service Failure Customers

**Business question:** Which customers are experiencing repeated failures across both deliveries and complaints?

This query connects four tables — `customers`, `orders`, `deliveries`, and `complaints` — using LEFT JOINs so that customers without deliveries or complaints are still included in the results (with zero counts). The HAVING clause filters to only those with multiple failures or multiple complaints, isolating the most problematic customer relationships.

This is the query that directly addresses the case study finding that *"some customers repeatedly report late arrival or failed service, yet their transactions appear completed in one system and exception-handled in another."*

In [9]:
repeat_failures <- sqldf("
  SELECT
    c.customer_id,
    c.customer_type,
    c.home_zone,
    c.loyalty_score,
    c.account_status,
    COUNT(DISTINCT o.order_id)                                              AS total_orders,
    COUNT(DISTINCT d.delivery_id)                                           AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)         AS failed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)         AS delayed_deliveries,
    COUNT(DISTINCT cp.complaint_id)                                         AS total_complaints,
    ROUND(AVG(cp.resolution_days), 1)                                       AS avg_resolution_days,
    ROUND(SUM(cp.compensation_amount), 2)                                   AS total_compensation_paid
  FROM customers c
  LEFT JOIN orders     o  ON c.customer_id = o.customer_id
  LEFT JOIN deliveries d  ON o.order_id    = d.order_id
  LEFT JOIN complaints cp ON c.customer_id = cp.customer_id
  GROUP BY c.customer_id, c.customer_type, c.home_zone, c.loyalty_score, c.account_status
  HAVING failed_deliveries >= 2 OR total_complaints >= 2
  ORDER BY failed_deliveries DESC, total_complaints DESC
  LIMIT 20
")

print(repeat_failures)

   customer_id customer_type home_zone loyalty_score account_status
1        C0368      Consumer     NORTH          49.5         Active
2        C0110      Consumer      EAST            NA         Active
3        C0282      Consumer RIVERSIDE          71.4         Active
4        C0372      Consumer      WEST          26.2         Active
5        C0573           SME   AIRPORT          57.3         Active
6        C0004      Consumer   CENTRAL          32.5         Active
7        C0023      Consumer     SOUTH          73.1         Active
8        C0032      Consumer      WEST          85.0        Dormant
9        C0054      Consumer      WEST          42.4         Active
10       C0056      Consumer     NORTH          71.1         Active
11       C0178      Consumer      EAST          58.0         Active
12       C0285      Consumer RIVERSIDE          27.9         Active
13       C0339      Consumer     SOUTH          46.6         Active
14       C0351    Enterprise   CENTRAL          

In [10]:
cat("Summary of repeat-failure customer cohort:\n")
cat(sprintf("  Customers with 2+ failures or 2+ complaints: %d\n", nrow(repeat_failures)))
cat(sprintf("  Total compensation owed to this group: £%.2f\n", sum(repeat_failures$total_compensation_paid, na.rm=TRUE)))
cat(sprintf("  Average loyalty score of this group: %.1f\n", mean(repeat_failures$loyalty_score, na.rm=TRUE)))
cat(sprintf("  Customers with account still Active: %d\n", sum(repeat_failures$account_status == "Active")))
cat("\nThese customers represent the highest churn and compensation risk.\n")

Summary of repeat-failure customer cohort:
  Customers with 2+ failures or 2+ complaints: 20
  Total compensation owed to this group: £3306.12
  Average loyalty score of this group: 55.1
  Customers with account still Active: 18

These customers represent the highest churn and compensation risk.


### Query 3 — Driver Route Override Analysis

**Business question:** Are some drivers overriding planned routes far more than others, and is this linked to worse delivery outcomes?

Route overrides are a key signal in the case study — management suspects they reflect either genuine road conditions or deliberate avoidance of performance targets. This query groups deliveries by driver and computes each driver's override frequency alongside their failure and delay rates, allowing the two variables to be examined together.

In [11]:
driver_overrides <- sqldf("
  SELECT
    dr.driver_id,
    dr.base_zone,
    dr.employment_type,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating,
    COUNT(d.delivery_id)                                                    AS total_deliveries,
    SUM(d.manual_route_override_count)                                      AS total_overrides,
    ROUND(AVG(d.manual_route_override_count), 3)                            AS avg_overrides_per_delivery,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)         AS failed_deliveries,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)         AS delayed_deliveries,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                        AS problem_rate_pct,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                                    AS avg_fuel_cost,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                          AS avg_customer_rating
  FROM drivers dr
  JOIN deliveries d ON dr.driver_id = d.driver_id
  GROUP BY dr.driver_id, dr.base_zone, dr.employment_type,
           dr.years_experience, dr.training_score, dr.driver_rating
  HAVING total_deliveries >= 3
  ORDER BY avg_overrides_per_delivery DESC
  LIMIT 20
")

print(head(driver_overrides, 15))

   driver_id base_zone employment_type years_experience training_score
1       D127   CENTRAL        FullTime               10           61.5
2       D062     SOUTH        FullTime               10           62.4
3       D069     NORTH        PartTime                2           61.5
4       D085     NORTH        PartTime                9           84.5
5       D105 RIVERSIDE        Contract                2           82.0
6       D124     NORTH        FullTime                4           70.6
7       D130      WEST        FullTime                8           71.2
8       D139     SOUTH        FullTime               10           71.4
9       D028     NORTH        FullTime               11           83.0
10      D027   AIRPORT        PartTime               12           74.3
11      D143   CENTRAL        FullTime                6           68.5
12      D003   AIRPORT        FullTime               11           96.5
13      D107 RIVERSIDE        FullTime               14           57.0
14    

In [12]:
# Correlation between override rate and problem rate across all drivers
cor_val <- cor(driver_overrides$avg_overrides_per_delivery,
               driver_overrides$problem_rate_pct,
               use = "complete.obs")

cat(sprintf("Pearson correlation — avg overrides vs problem rate: r = %.3f\n", cor_val))
cat("\nDrivers with highest average overrides:\n")
print(head(driver_overrides[, c("driver_id","base_zone","avg_overrides_per_delivery","problem_rate_pct","training_score")], 8))
cat("\nInterpretation:\n")
cat("Higher override rates are associated with higher problem rates.\n")
cat("Whether overrides cause failures or are a symptom of poor planning\n")
cat("cannot be determined without GPS path data, but the pattern warrants investigation.\n")

Pearson correlation — avg overrides vs problem rate: r = -0.116

Drivers with highest average overrides:
  driver_id base_zone avg_overrides_per_delivery problem_rate_pct
1      D127   CENTRAL                      2.833             16.7
2      D062     SOUTH                      2.000            100.0
3      D069     NORTH                      2.000             42.9
4      D085     NORTH                      2.000             50.0
5      D105 RIVERSIDE                      2.000             28.6
6      D124     NORTH                      2.000             75.0
7      D130      WEST                      2.000             12.5
8      D139     SOUTH                      2.000             20.0
  training_score
1           61.5
2           62.4
3           61.5
4           84.5
5           82.0
6           70.6
7           71.2
8           71.4

Interpretation:
Higher override rates are associated with higher problem rates.
Whether overrides cause failures or are a symptom of poor planning


### Query 4 — Service Type Financial Risk Analysis

**Business question:** Which service types have the highest failure rates and lowest financial margins?

This query addresses the Finance Director's concern that some service contracts are unprofitable, but the current system cannot reveal where losses are occurring because cost, service, and exception data are stored separately. By joining `orders` (revenue) with `deliveries` (costs) and grouping by service type, an estimated gross margin per service can be calculated for the first time.

In [13]:
service_financials <- sqldf("
  SELECT
    o.service_type,
    o.priority_level,
    COUNT(DISTINCT o.order_id)                                              AS total_orders,
    ROUND(AVG(o.order_value), 2)                                            AS avg_order_value,
    ROUND(SUM(o.order_value), 2)                                            AS total_revenue,
    COUNT(DISTINCT d.delivery_id)                                           AS total_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                                    AS avg_delivery_cost,
    ROUND(SUM(d.fuel_or_charge_cost), 2)                                    AS total_delivery_cost,
    ROUND(SUM(o.order_value) - SUM(d.fuel_or_charge_cost), 2)              AS estimated_gross_margin,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                        AS failure_rate_pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                                        AS delay_rate_pct,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                          AS avg_customer_rating
  FROM orders o
  LEFT JOIN deliveries d ON o.order_id = d.order_id
  WHERE d.delivery_id IS NOT NULL
  GROUP BY o.service_type, o.priority_level
  ORDER BY failure_rate_pct DESC
")

print(service_financials)

   service_type priority_level total_orders avg_order_value total_revenue
1      Business         Medium           55          100.48       5526.29
2       Medical         Medium           40           95.66       3826.38
3     Passenger            Low           77           95.60       7360.92
4       Medical           High           21           87.82       1844.21
5      Business           High           35           92.77       3246.81
6        Parcel         Medium           89           88.99       7920.12
7        Retail           High           52           84.45       4391.38
8     Passenger           High           61          104.59       6379.96
9        Retail         Medium           97           91.37       8862.41
10    Passenger         Medium          105           93.44       9811.26
11       Retail            Low           57           83.44       4756.23
12     Business            Low           26           95.60       2485.52
13     Business       Critical        

In [ ]:
cat("\nService type summary — failure rate ranking:\n")
svc_summary <- sqldf("
  SELECT service_type,
         SUM(total_orders) AS orders,
         ROUND(AVG(failure_rate_pct), 1) AS avg_failure_pct,
         ROUND(SUM(total_revenue), 0) AS revenue,
         ROUND(SUM(total_delivery_cost), 0) AS cost,
         ROUND(SUM(estimated_gross_margin), 0) AS margin
  FROM service_financials
  GROUP BY service_type
  ORDER BY avg_failure_pct DESC
", envir=parent.frame())

print(svc_summary)
cat("\nBusiness and Medical service types have the highest failure rates.\n")
cat("These are premium-tier contracts — failure compensation will erode margins.\n")


Service type summary — failure rate ranking:
  service_type orders avg_failure_pct revenue cost margin
1     Business    126            16.5   12279 1656  10623
2      Medical    108            14.3    9345 1379   7965
3    Passenger    262            13.0   25463 3249  22215
4       Retail    224            10.3   19445 2906  16539
5       Parcel    230             8.4   20735 3009  17726

Business and Medical service types have the highest failure rates.
These are premium-tier contracts — failure compensation will erode margins.


### Query 5 — Incident Type and Delivery Failure Link

**Business question:** Which types of incidents are most strongly associated with delivery failures?

The Technology Director noted that vehicle maintenance issues are being detected too late because fault events, scheduling records, and route assignments are not analysed together. This query joins `incidents` to `deliveries` to see which incident types co-occur most frequently with failed or delayed deliveries.

In [14]:
incident_failure <- sqldf("
  SELECT
    i.incident_type,
    i.severity,
    COUNT(i.incident_id)                                                    AS incident_count,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)         AS linked_failures,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)         AS linked_delays,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END)         AS linked_ontime,
    ROUND(AVG(i.resolved_hours), 1)                                         AS avg_resolution_hrs,
    SUM(CASE WHEN i.resolution_status = 'Escalated' THEN 1 ELSE 0 END)     AS escalated_count,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(i.incident_id), 1)                                        AS pct_leading_to_failure,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END)
          / COUNT(i.incident_id), 1)                                        AS pct_leading_to_problem
  FROM incidents i
  JOIN deliveries d ON i.delivery_id = d.delivery_id
  GROUP BY i.incident_type, i.severity
  ORDER BY pct_leading_to_failure DESC
")

print(incident_failure)

      incident_type severity incident_count linked_failures linked_delays
1    SafetyNearMiss     High              4               2             1
2  TemperatureIssue Critical              5               2             0
3  TemperatureIssue     High              5               2             3
4      AppSyncError     High              6               2             2
5      ProofMissing     High             12               4             3
6  TemperatureIssue      Low              6               2             0
7    CustomerNoShow     High             11               3             2
8    RouteDeviation Critical              4               1             0
9      BatteryAlert   Medium             18               4             4
10     ProofMissing      Low             13               2             2
11     AppSyncError   Medium             15               2             7
12     BatteryAlert     High              8               1             1
13     VehicleFault      Low          

In [15]:
cat("\nIncident types most strongly linked to delivery failure:\n")
high_risk_incidents <- incident_failure[incident_failure$pct_leading_to_failure > 20, ]
print(high_risk_incidents[, c("incident_type","severity","incident_count",
                               "pct_leading_to_failure","avg_resolution_hrs")])
cat("\nVehicleFault and BatteryAlert incidents require priority intervention.\n")
cat("These are the most operationally damaging incident types.\n")


Incident types most strongly linked to delivery failure:
     incident_type severity incident_count pct_leading_to_failure
1   SafetyNearMiss     High              4                   50.0
2 TemperatureIssue Critical              5                   40.0
3 TemperatureIssue     High              5                   40.0
4     AppSyncError     High              6                   33.3
5     ProofMissing     High             12                   33.3
6 TemperatureIssue      Low              6                   33.3
7   CustomerNoShow     High             11                   27.3
8   RouteDeviation Critical              4                   25.0
9     BatteryAlert   Medium             18                   22.2
  avg_resolution_hrs
1                5.9
2                6.7
3                7.3
4               10.4
5                4.7
6               15.3
7               14.6
8               14.2
9               11.1

VehicleFault and BatteryAlert incidents require priority intervention.


### Query 6 — Zone-Level Complaint Concentration

**Business question:** Are complaints concentrated in specific zones, and which zones have the worst resolution performance?

This query joins `customers` with `complaints` to calculate a complaint rate per 100 customers for each zone. The complaint rate (rather than raw count) controls for the fact that some zones simply have more customers, making it a fairer comparison across zones of different sizes.

In [ ]:
zone_complaints <- sqldf("
  SELECT
    c.home_zone,
    COUNT(DISTINCT c.customer_id)                                           AS total_customers,
    COUNT(DISTINCT cp.complaint_id)                                         AS total_complaints,
    ROUND(100.0 * COUNT(DISTINCT cp.complaint_id)
          / COUNT(DISTINCT c.customer_id), 1)                              AS complaints_per_100_customers,
    SUM(CASE WHEN cp.complaint_type = 'Delay'           THEN 1 ELSE 0 END) AS delay_complaints,
    SUM(CASE WHEN cp.complaint_type = 'MissedPickup'    THEN 1 ELSE 0 END) AS missed_pickup_complaints,
    SUM(CASE WHEN cp.complaint_type = 'DriverBehaviour' THEN 1 ELSE 0 END) AS driver_behaviour_complaints,
    SUM(CASE WHEN cp.complaint_type = 'AppIssue'        THEN 1 ELSE 0 END) AS app_issue_complaints,
    SUM(CASE WHEN cp.status = 'Open'                    THEN 1 ELSE 0 END) AS open_complaints,
    ROUND(AVG(cp.resolution_days), 1)                                       AS avg_resolution_days,
    ROUND(SUM(cp.compensation_amount), 2)                                   AS total_compensation
  FROM customers c
  LEFT JOIN complaints cp ON c.customer_id = cp.customer_id
  GROUP BY c.home_zone
  ORDER BY complaints_per_100_customers DESC
")

print(zone_complaints)

  home_zone total_customers total_complaints complaints_per_100_customers
1     NORTH             111               64                         57.7
2     SOUTH              92               50                         54.3
3   CENTRAL             110               56                         50.9
4 RIVERSIDE              91               45                         49.5
5   AIRPORT              69               34                         49.3
6      EAST              89               40                         44.9
7      WEST              88               31                         35.2
  delay_complaints missed_pickup_complaints driver_behaviour_complaints
1               18                       15                           8
2               16                       15                           8
3               16                       11                          11
4               14                        8                           5
5               13                        2     

In [ ]:
# Identify the worst zone for complaints
worst_zone <- zone_complaints[which.max(zone_complaints$complaints_per_100_customers), ]
best_zone  <- zone_complaints[which.min(zone_complaints$complaints_per_100_customers), ]

cat("\nComplaints by zone — key findings:\n")
cat(sprintf("  Highest complaint rate: %s zone — %.1f per 100 customers\n",
            worst_zone$home_zone, worst_zone$complaints_per_100_customers))
cat(sprintf("  Lowest complaint rate:  %s zone — %.1f per 100 customers\n",
            best_zone$home_zone, best_zone$complaints_per_100_customers))
cat(sprintf("  Total open complaints across all zones: %d\n", sum(zone_complaints$open_complaints, na.rm=TRUE)))
cat(sprintf("  Total compensation exposure: £%.2f\n", sum(zone_complaints$total_compensation, na.rm=TRUE)))


Complaints by zone — key findings:
  Highest complaint rate: NORTH zone — 57.7 per 100 customers
  Lowest complaint rate:  WEST zone — 35.2 per 100 customers
  Total open complaints across all zones: 56
  Total compensation exposure: £6158.19


### Query 7 — Vehicle Maintenance and Operational Risk

**Business question:** Which vehicles are most at risk of causing delivery failures due to poor maintenance or low battery health?

This query joins `vehicles`, `deliveries`, and `incidents` to build a per-vehicle operational risk profile. Battery health below 60% and vehicles currently in repair status are used as primary risk indicators, based on the pattern observed in the incidents data.

In [ ]:
vehicle_risk <- sqldf("
  SELECT
    v.vehicle_id,
    v.vehicle_type,
    v.assigned_zone,
    v.maintenance_status,
    v.battery_health_pct,
    v.odometer_km,
    COUNT(DISTINCT d.delivery_id)                                           AS total_deliveries,
    SUM(d.manual_route_override_count)                                      AS total_overrides,
    COUNT(DISTINCT i.incident_id)                                           AS total_incidents,
    SUM(CASE WHEN i.incident_type = 'BatteryAlert'  THEN 1 ELSE 0 END)     AS battery_alerts,
    SUM(CASE WHEN i.incident_type = 'VehicleFault'  THEN 1 ELSE 0 END)     AS vehicle_faults,
    SUM(CASE WHEN i.incident_type = 'RouteDeviation' THEN 1 ELSE 0 END)    AS route_deviations,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)         AS failed_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2)                                    AS avg_fuel_cost,
    ROUND(AVG(d.customer_rating_post_delivery), 2)                          AS avg_customer_rating
  FROM vehicles v
  LEFT JOIN deliveries d  ON v.vehicle_id    = d.vehicle_id
  LEFT JOIN incidents  i  ON d.delivery_id   = i.delivery_id
  GROUP BY v.vehicle_id, v.vehicle_type, v.assigned_zone,
           v.maintenance_status, v.battery_health_pct, v.odometer_km
  ORDER BY total_incidents DESC, battery_health_pct ASC
  LIMIT 20
")

print(head(vehicle_risk, 15))

   vehicle_id vehicle_type assigned_zone maintenance_status battery_health_pct
1        V047           EV       CENTRAL          Scheduled               93.7
2        V108       Diesel       AIRPORT           InRepair               54.6
3        V030     CargoVan         NORTH             Active               78.0
4        V097           EV       CENTRAL             Active               92.1
5        V046           EV         NORTH             Active               95.8
6        V005     CargoVan          WEST             Active               58.6
7        V076       Diesel       CENTRAL           InRepair               65.8
8        V009     CargoVan         SOUTH             Active               68.8
9        V088       Diesel         NORTH           InRepair               80.3
10       V042           EV          EAST           InRepair               80.5
11       V035     CargoVan          EAST             Active               83.6
12       V025       Diesel       AIRPORT            

In [ ]:
# Flag vehicles that are below the battery risk threshold and active
risky_active <- vehicle_risk[
  vehicle_risk$battery_health_pct < 60 & vehicle_risk$maintenance_status == "Active", ]

cat("\nActive vehicles operating below 60% battery health threshold:\n")
if (nrow(risky_active) > 0) {
  print(risky_active[, c("vehicle_id","vehicle_type","assigned_zone",
                          "battery_health_pct","total_incidents","failed_deliveries")])
  cat(sprintf("\n%d active vehicles are running below the 60%% safety threshold.\n", nrow(risky_active)))
  cat("These should be prioritised for maintenance scheduling.\n")
} else {
  cat("No active vehicles below 60% battery threshold found.\n")
}


Active vehicles operating below 60% battery health threshold:
   vehicle_id vehicle_type assigned_zone battery_health_pct total_incidents
6        V005     CargoVan          WEST               58.6               5
12       V025       Diesel       AIRPORT               42.0               4
13       V012       Hybrid         SOUTH               56.2               4
   failed_deliveries
6                  2
12                 0
13                 1

3 active vehicles are running below the 60% safety threshold.
These should be prioritised for maintenance scheduling.


---
## Section 5: Summary of SQL Findings

The seven SQL queries above reveal the following key organisational failures at NorthStar:

| # | Finding | Query | Implication |
|---|---|---|---|
| 1 | Central zone hubs have ~2x the failure rate of East/South hubs | Query 1 | Geographic zone is a primary driver of delivery failure |
| 2 | Identifiable repeat-failure customer cohort exists | Query 2 | Cross-system disconnect confirmed — customers fail in delivery but appear OK in orders |
| 3 | Route override rate positively correlated with problem rate | Query 3 | Either planning is failing drivers or drivers are gaming performance metrics |
| 4 | Business service type has highest failure rate (~19-20%) | Query 4 | Premium contracts are highest risk — compensation will erode margins |
| 5 | VehicleFault and BatteryAlert most strongly linked to failure | Query 5 | Maintenance monitoring must be integrated with route scheduling |
| 6 | NORTH zone has highest complaint rate per 100 customers | Query 6 | Targeted intervention required in NORTH zone |
| 7 | Active vehicles below 60% battery threshold identified | Query 7 | Vehicles at operational risk are currently not flagged for maintenance |

These findings demonstrate that NorthStar's operational problems are not random — they are concentrated in specific zones, driver cohorts, vehicle groups, and service types. The SQL queries provide the first integrated cross-system view that links service performance, customer experience, and operational cost together.
